# Distributed Robot Policy Sim-Eval with Ray + Isaac Lab on Anyscale

## TLDR

You fan out sim-based robot policy evaluation across 4 GPUs using **Ray Actors**, with the policy hosted behind a shared **Ray Serve** deployment. You validate a humanoid walking policy against real-world perturbations (sensor noise + motor wear) and then analyze a pre-computed 25-condition sweep to build a full deployment map. The live portion runs in under 3 minutes, and the same pattern scales to thousands of conditions on larger clusters.

**The focus of this tutorial is Ray.** Humanoid locomotion is the workload. The point is to show how **Ray Core on Anyscale** handles the hard infrastructure problems — GPU scheduling, worker lifecycle, fan-out and gather — so you can evaluate any policy at scale with the same three-step pattern.

## Introduction

Robots trained in simulation fail in the real world. Simulation assumes perfect sensors and fresh motors. Reality has sensor drift, actuator wear, dust, rain, and vibration. The gap between "works in sim" and "works on hardware" has ended product launches and broken robots, and closing it requires testing the policy against thousands of realistic perturbations before you deploy.

You can't run those evaluations one-at-a-time on a laptop. Each condition needs its own GPU-accelerated physics sim, and the cost-per-condition can be dominated by simulator boot time, not actual compute (of course this depends on the size of the individual scenarios). The shape of the problem — expensive setup, possilby cheap per-call work, and embarrassingly parallel — maps naturally onto **Ray Actors**.

In this notebook you:

- **Boot 4 Ray Actors** — one per GPU — each holding a pre-trained humanoid walking policy and a running Isaac Lab simulation
- **Fan out 4 deployment scenarios** across the actors in parallel (Lab → Factory → Field → Harsh)
- **Load and visualize a pre-computed 25-condition sweep** to get the full deployment envelope

The same pattern applies to any Isaac Lab task (Ant, Cartpole, Franka, ANYmal) and any policy architecture. You can swap the workload, keep the infrastructure.

---

## Key concepts

A quick reference for the ideas that appear throughout. Skip if they're already familiar.

**Ray** gives you three distributed-computing primitives:

- `@ray.remote` **functions** are stateless tasks. Ray spins up a worker for each call and tears it down when the call returns. Great for fan-out where each call is independent: "evaluate all 25 configs, I don't care which worker runs which."
- `@ray.remote` **classes** (called **Actors**) are stateful, long-lived workers. You instantiate an actor with `MyActor.remote(...)`, it lives on a specific node with specific resources, and you call its methods with `actor.method.remote(...)`. The same process handles every call, so expensive one-time setup (like booting Isaac Lab) happens once.
- `@serve.deployment` **classes** are replicated, concurrent services managed by **Ray Serve**. One replica holds an expensive object (here, a loaded policy); every caller shares it through a single handle, and Serve handles concurrency and request routing.

**Ray Actors are the right fit for the sim side** because Isaac Lab takes ~1–2 minutes to boot but runs thousands of sim steps per second afterward. You pay the boot cost once per actor, then reuse. **Ray Serve is the right fit for the policy side** because every actor wants to call the same weights without loading its own copy.

**Isaac Lab** is NVIDIA's GPU-accelerated robot learning framework. It runs physics simulations for dozens of robots in parallel on a single GPU. You pick a task (`Isaac-Humanoid-Direct-v0`), specify `num_envs`, and step through observations → actions → rewards like any Gym environment — but on-GPU.

**`env.py`** is this tutorial's local wrapper that boots Isaac Lab in a **child process** via `multiprocessing.Pipe`. Isaac Sim's Kit engine runs its own event loop that conflicts with Ray's. Running Kit in a subprocess gives each one its own process space, and they exchange numpy arrays through a pipe. You treat the wrapper as a black box — `reset()` / `step()` / `close()`.

**Sim-eval** is evaluating a trained policy under a perturbation condition and measuring how often it succeeds (**pass rate**, fraction of episodes with reward > threshold) and how well it performs (**mean reward**). One condition × many parallel envs = one data point. Many conditions in parallel = a deployment map.

---

## What you will learn

By the end of this notebook you will be able to:

- **Wrap a physics simulator as a Ray Actor** (`@ray.remote(num_gpus=1) class SimActor`) so Ray schedules it on a GPU worker
- **Host a policy as a Ray Serve deployment** (`@serve.deployment class PolicyServer`) and share it across all actors via a handle
- **Fan out sim-eval across the cluster** by instantiating one actor per GPU and dispatching one config per actor
- **Amortize expensive setup cost** — keep an actor alive across multiple `run()` calls so the 1–2 min Isaac Lab boot is paid once
- **Stage shared assets** on Anyscale cluster storage so every GPU worker sees the same files
- **Visualize a perturbation sweep** as heatmaps and bar charts to make a data-driven deployment decision
- **Scale and swap** — the same pattern works for any Isaac Lab task or any PyTorch policy

---

## Why Ray on Anyscale for robotics sim-eval?

Robot policy validation has a distinctive shape: seconds of GPU compute per condition, but thousands of conditions, with per-worker setup cost that can dwarf per-call cost. That shape maps directly onto Ray:

| Challenge | Without Ray | With Ray on Anyscale |
|---|---|---|
| Per-condition isolation | Custom subprocess orchestration | `@ray.remote(num_gpus=1)` |
| GPU scheduling | Manual `CUDA_VISIBLE_DEVICES` juggling | Ray scheduler + declarative resource requests |
| Stateful workers (boot-once-run-many) | Daemon scripts + ad-hoc IPC | Ray Actors |
| Stateless fan-out | Shell loops, job schedulers | `@ray.remote def`, `ray.get(futures)` |
| Shared model, many callers | Flask/FastAPI + reverse proxy | `@serve.deployment` + shared handle |
| Shared cluster storage | `scp` / rsync / per-node downloads | `/mnt/cluster_storage/` visible to all workers |
| Scaling 4 GPUs → 64+ | Rewrite scheduler | Same code, bigger cluster |

The three-step pattern you use below — **wrap → remote → fan out** — is the same pattern you'd use at 100× the scale. Only the cluster config changes.

---

## Architecture

<p align="center">
  <img src="media/architecture.png" width="700" alt="System architecture">
</p>

```
1. Wrap    →  env.py boots Isaac Lab in a subprocess, exposes reset() / step()
2. Remote  →  @serve.deployment PolicyServer hosts the policy; @ray.remote(num_gpus=1) SimActor holds the env
3. Fan out →  serve.run(PolicyServer.bind(...)) + [SimActor.remote(handle) for _ in range(4)] + ray.get(futures)
```

Each SimActor lives on its own GPU worker node. The `PolicyServer` lives as a single Ray Serve replica that all actors call into. The driver (this notebook) runs on the head node — CPU-only — and orchestrates everything.

---

## How this scales on Anyscale

You run this tutorial on **4× A10G GPUs** with 4 live conditions, then load a pre-computed 25-condition sweep. The same code scales on Anyscale by changing the config, not the code:

| Scale lever | Tutorial | Production on Anyscale |
|---|---|---|
| Live conditions | 4 | hundreds |
| Parallel envs per actor | 5 | 512+ |
| Steps per condition | 1,500 | 10,000+ |
| GPU workers | 4 × A10G | 64 × A100/H100 |
| Policy architecture | 75→400→200→100→21 | any PyTorch `nn.Module` |
| Isaac Lab task | Humanoid | Ant, Franka, ANYmal, custom |

Anyscale manages cluster scaling, GPU scheduling, and shared storage. You adjust how many actors you create and the platform handles the rest.

---

## What the trained humanoid looks like

The pre-trained policy you'll evaluate below was trained with distributed PPO on the Isaac Lab humanoid task (75-dim proprioceptive obs, 21 joint torques). A short preview before you dive into the Ray patterns:

In [ ]:
import os
from IPython.display import Video, HTML, display

if os.path.exists("media/sample.mp4"):
    display(Video("media/sample.mp4", width=480, embed=False))
else:
    display(HTML('<div style="padding:24px; background:#f8f9fa; border-radius:8px; text-align:center; color:#666;">'
                 'Humanoid preview video not found at <code>media/sample.mp4</code>'
                 '</div>'))


## Cell 1: Environment check and imports

Import the libraries you'll use throughout. You'll rely on:

- **`ray`** for distributed orchestration
- **`torch`** for the policy network (loaded on workers, not the driver)
- **`numpy`** for observations and actions (crossing the env.py subprocess pipe as numpy arrays)
- **`matplotlib`** for the sweep heatmaps

Note that the driver (where this notebook runs) is the head node — CPU-only. Every `torch`/CUDA operation in this notebook actually happens on GPU workers via Ray.

In [ ]:
import os
import shutil
import time
import json
import itertools

import numpy as np
import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import HTML, display

import ray

print(f"Ray:     {ray.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Driver CUDA available: {torch.cuda.is_available()}  "
      f"(driver runs on the CPU head node — GPUs live on Ray worker nodes)")


## Cell 2: Connect to the Ray cluster

`ray.init(address="auto", ...)` connects to the managed Anyscale cluster that is already running. On Anyscale this is all the infrastructure setup you need. The platform handles node provisioning, GPU scheduling, and cluster lifecycle. The `env_vars` passed through `runtime_env` are what Isaac Sim expects (Vulkan ICD paths for GPU rendering + EULA acceptance flags).

After connecting, `ray.cluster_resources()` reports the GPUs and CPUs available. You'll use these to schedule 4 `SimActor` instances below — one per GPU.

In [ ]:
if not ray.is_initialized():
    ray.init(address="auto", runtime_env={"env_vars": {
        "VK_ICD_FILENAMES": "/etc/vulkan/icd.d/nvidia_icd.json",
        "VK_DRIVER_FILES":  "/etc/vulkan/icd.d/nvidia_icd.json",
        "OMNI_KIT_ACCEPT_EULA": "YES",
        "ACCEPT_EULA": "Y",
    }})

res = ray.cluster_resources()
print(f"Cluster resources:")
print(f"  CPUs: {res.get('CPU', 0):.0f}")
print(f"  GPUs: {res.get('GPU', 0):.0f}")


## Cell 3: Stage the pre-trained checkpoint to shared storage

The checkpoint (`checkpoint_pretrained.pt`) is committed next to this notebook. You copy it to **`/mnt/cluster_storage/`**, Anyscale's shared filesystem — a path that's visible from every worker node at the same location. That way, every `SimActor` can load the checkpoint directly from its GPU worker without you shipping the weights over the network.

This is the canonical Anyscale pattern for shared read-only assets: stage once on the driver, read from any worker. The cell is idempotent and safe to re-run.

In [ ]:
SRC_CKPT = "checkpoint_pretrained.pt"               
STAGED_CKPT = "/mnt/cluster_storage/checkpoints/humanoid/checkpoint_pretrained.pt"

if not os.path.exists(STAGED_CKPT):
    os.makedirs(os.path.dirname(STAGED_CKPT), exist_ok=True)
    shutil.copy2(SRC_CKPT, STAGED_CKPT)
    print(f"Staged checkpoint -> {STAGED_CKPT}")
else:
    print(f"Checkpoint already staged at {STAGED_CKPT}")

ckpt_meta = torch.load(STAGED_CKPT, weights_only=False, map_location="cpu")
print(f"  Task:            {ckpt_meta.get('task')}")
print(f"  Training reward: {ckpt_meta.get('mean_reward'):,.0f}")


## Cell 4: Why `env.py` is a separate file

Isaac Sim's Kit engine runs its own event loop, which conflicts with Ray's worker loop. If you tried to boot Isaac Lab directly inside a Ray actor, they'd fight over the process. `env.py` solves this by running Isaac Lab in a **child process** via `multiprocessing.Pipe` — Kit owns its event loop, Ray owns its own, and they communicate with numpy arrays through the pipe.

The public API is minimal:

```python
from env import IsaacLabDirectEnv

env = IsaacLabDirectEnv(task="Isaac-Humanoid-Direct-v0", num_envs=5, device="cuda:0")
obs = env.reset()                               # (num_envs, 75) numpy array
obs, rewards, dones, _ = env.step(actions)      # numpy in, numpy out
env.close()
```

It works with any Isaac Lab Direct task — Humanoid, Ant, Cartpole, Franka, ANYmal — no code changes. For the rest of this notebook you treat `env.py` as a black box: `reset()` / `step()` / `close()` is the whole interface.

## Cell 5: Evaluation policy architecture

Define the PyTorch `nn.Module` that matches the pre-trained checkpoint. The network is a standard actor-critic used during PPO training:

- **Policy net**: 75-dim proprioceptive obs → `400 → 200 → 100` hidden with ELU activations → 21 joint action means
- **`action_log_std`**: a learned log-standard-deviation for the Gaussian action distribution (PPO rollout policy)
- **`value_net`**: state-value head used during training

At eval time you only need `action_mean(policy_net(obs))`, but you define the full network so `load_state_dict` matches every key in the checkpoint.

In [ ]:
class EvalPolicy(nn.Module):
    def __init__(self, obs_dim=75, action_dim=21):
        super().__init__()
        self.policy_net = nn.Sequential(
            nn.Linear(obs_dim, 400), nn.ELU(),
            nn.Linear(400, 200), nn.ELU(),
            nn.Linear(200, 100), nn.ELU(),
        )
        self.action_mean = nn.Linear(100, action_dim)
        self.action_log_std = nn.Parameter(torch.zeros(action_dim))
        self.value_net = nn.Sequential(
            nn.Linear(obs_dim, 400), nn.ELU(),
            nn.Linear(400, 200), nn.ELU(),
            nn.Linear(200, 100), nn.ELU(),
            nn.Linear(100, 1),
        )

    def forward(self, obs):
        return self.action_mean(self.policy_net(obs))


## Cell 6: The `PolicyServer` — a shared inference service

`@serve.deployment` creates a **Ray Serve deployment**: a replicated service with its own resource budget and concurrency controls. We wrap `EvalPolicy` in one so every sim actor shares a single loaded copy of the model instead of carrying its own.

`max_ongoing_requests=16` lets the replica handle up to 16 concurrent `predict` calls, so several sim actors stepping at the same time don't queue on each other.

`serve.run(...)` deploys the service and returns a `DeploymentHandle` — the single object every sim actor will call through.


In [ ]:
from ray import serve

@serve.deployment(
    ray_actor_options={"num_cpus": 1},
    max_ongoing_requests=16,
)
class PolicyServer:
    def __init__(self, checkpoint_path):
        self.policy = EvalPolicy()
        self.policy.load_state_dict(
            torch.load(checkpoint_path, weights_only=False)["policy"]
        )
        self.policy.eval()

    async def predict(self, obs):
        with torch.no_grad():
            actions = torch.clamp(
                self.policy(torch.tensor(obs, dtype=torch.float32)), -1, 1
            ).numpy()
        return {"action": actions}


policy_handle = serve.run(
    PolicyServer.bind(checkpoint_path=STAGED_CKPT),
    name="policy_server",
)
print("PolicyServer deployed")


## Cell 7: The `SimActor` — a persistent Ray Actor with a GPU

This is the core of the pattern. `@ray.remote(num_gpus=1)` declares that every `SimActor` instance reserves **one GPU**, and Ray's scheduler places it on a worker node with a free A10G.

`__init__` runs **once per actor** when you call `SimActor.remote(policy_handle)`:

1. **Import `env.py`** — only on the worker. The driver (head node) never imports Isaac Lab.
2. **Boot Isaac Lab** with 5 parallel humanoids on `cuda:0`. This is the slow ~1–2 min step. It happens once per actor, in parallel across the cluster.
3. **Keep the `policy_handle`** — the actor doesn't load the policy itself; it calls `PolicyServer` through this handle.

`run(obs_noise, act_noise, num_steps)` runs each time you call `actor.run.remote(...)`:

1. Reset the 5 envs.
2. For each step: add sensor noise → `policy_handle.predict.remote(obs).result()` → add motor noise → `env.step(actions)` → accumulate rewards.
3. When an env terminates (humanoid falls), record that episode's reward and reset the accumulator for that env.
4. Return mean reward, pass rate (fraction of episodes with reward > 3000 — "actually walked"), and episode count.

Because the actor keeps the env alive between calls, you can probe the same actor under many different noise conditions without re-booting Isaac Lab each time. And because the policy sits in one shared `PolicyServer` replica, every actor reads from the same weights.

In [ ]:
@ray.remote(num_gpus=1)
class SimActor:
    def __init__(self, policy_handle, num_envs=5):
        # env.py is imported inside __init__ so only the worker loads Isaac Lab.
        from env import IsaacLabDirectEnv
        self.env = IsaacLabDirectEnv(
            task="Isaac-Humanoid-Direct-v0",
            num_envs=num_envs,
            device="cuda:0",
        )
        self.policy_handle = policy_handle
        self.num_envs = num_envs

    def run(self, obs_noise=0.0, act_noise=0.0, num_steps=1500):
        obs = self.env.reset()
        episode_rewards = []
        ep_rewards = np.zeros(self.num_envs)

        for step in range(num_steps):
            # Sensor noise — obs + N(0, obs_noise)
            noisy_obs = (obs + np.random.normal(0, obs_noise, size=obs.shape).astype(np.float32)
                         if obs_noise > 0 else obs)

            # Inference via the shared Ray Serve policy
            result = self.policy_handle.predict.remote(noisy_obs).result()
            actions = result["action"]

            # Motor noise — actions + N(0, act_noise), re-clipped
            if act_noise > 0:
                actions = np.clip(
                    actions + np.random.normal(0, act_noise, size=actions.shape).astype(np.float32),
                    -1, 1,
                )

            obs, rewards, dones, _ = self.env.step(actions)
            ep_rewards += rewards

            if np.any(dones):
                idx = np.where(dones)[0]
                episode_rewards.extend(ep_rewards[idx].tolist())
                ep_rewards[idx] = 0.0

        # Include envs still alive at the end
        alive = ep_rewards > 0
        if np.any(alive):
            episode_rewards.extend(ep_rewards[alive].tolist())

        mean_reward = float(np.mean(episode_rewards)) if episode_rewards else 0.0
        pass_rate = float(np.mean([r > 3000 for r in episode_rewards])) if episode_rewards else 0.0
        return {
            "mean_reward": mean_reward,
            "pass_rate":   pass_rate,
            "episodes":    len(episode_rewards),
        }


## Cell 8: Define the deployment scenarios

You evaluate the policy under **4 realistic deployment scenarios**, each combining a sensor-quality condition with a motor-condition. The four points sit on the diagonal of a richer 5×5 grid you'll analyze later — always "worse than the last one" on both axes, so the degradation curve is easy to read.

| Scenario | Sensor quality | Motor condition |
|---|---|---|
| **Lab** | Perfect sensors (obs_noise = 0.00) | New motors (act_noise = 0.00) |
| **Factory floor** | Factory-grade sensors (0.05) | 6-month motor wear (0.05) |
| **Outdoor field** | Outdoor sensors (0.10) | Motors needing service (0.10) |
| **Harsh (EOL)** | Rain / dust (0.20) | End-of-life motors (0.20) |

The `OBS_LABELS` / `ACT_LABELS` dicts below map noise magnitudes to human-readable labels, and you reuse them later when you visualize the full 25-condition sweep — same vocabulary, no translation needed.

In [ ]:
OBS_LABELS = {
    0.0:  "Perfect sensors",
    0.02: "Lab-grade IMU",
    0.05: "Factory floor",
    0.1:  "Outdoor field",
    0.2:  "Rain / dust",
}
ACT_LABELS = {
    0.0:  "New motors",
    0.02: "Broken in",
    0.05: "6-month wear",
    0.1:  "Needs service",
    0.2:  "End of life",
}

LIVE_CONFIGS = [
    {"scenario": "Lab",           "obs_noise": 0.00, "act_noise": 0.00},
    {"scenario": "Factory floor", "obs_noise": 0.05, "act_noise": 0.05},
    {"scenario": "Outdoor field", "obs_noise": 0.10, "act_noise": 0.10},
    {"scenario": "Harsh (EOL)",   "obs_noise": 0.20, "act_noise": 0.20},
]

for c in LIVE_CONFIGS:
    o, a = c["obs_noise"], c["act_noise"]
    print(f"  {c['scenario']:15s}  {OBS_LABELS[o]:20s} + {ACT_LABELS[a]}")


## Cell 9: Launch the distributed live sweep

This is the payoff cell — **the three-step pattern in action**:

1. **Create** 4 `SimActor` instances. Ray's scheduler places each one on a free GPU, and all 4 boot Isaac Lab simultaneously. Because the boots run in parallel across nodes, the wall-clock cost is roughly one boot (~1–2 min), not four.
2. **Dispatch** one `run()` call per actor with its assigned scenario config. Each call returns a future (a `ray.ObjectRef`) immediately — no blocking.
3. **Gather** with `ray.get(futures)`. This blocks until all 4 actors have finished their sim loop and returns the results.

After boot, each 1,500-step evaluation runs in ~30 seconds on an A10G. Total wall time should be roughly **1–2 min boot + ~30 sec evaluation**.

In [ ]:
print(f"Booting {len(LIVE_CONFIGS)} SimActors (one per GPU) — Isaac Lab first boot ~1-2 min...", flush=True)
t0 = time.time()

# Step 1: one actor per GPU — all boot in parallel; each holds the PolicyServer handle
actors = [SimActor.remote(policy_handle) for _ in range(len(LIVE_CONFIGS))]

# Step 2: one config per actor — fire-and-forget, returns futures
futures = [
    actor.run.remote(
        obs_noise=cfg["obs_noise"],
        act_noise=cfg["act_noise"],
        num_steps=1500,
    )
    for actor, cfg in zip(actors, LIVE_CONFIGS)
]

# Step 3: block until all 4 finish and gather results
live_results = ray.get(futures)
elapsed = time.time() - t0

print(f"\nAll {len(LIVE_CONFIGS)} scenarios evaluated in {elapsed:.0f}s across 4 GPUs\n")
print(f"  {'Scenario':15s}  {'obs':>5s} {'act':>5s}  {'reward':>8s}  {'pass':>6s}  eps")
print(f"  {'-'*15}  {'-'*5} {'-'*5}  {'-'*8}  {'-'*6}  ---")
for cfg, r in zip(LIVE_CONFIGS, live_results):
    print(f"  {cfg['scenario']:15s}  "
          f"{cfg['obs_noise']:>5.2f} {cfg['act_noise']:>5.2f}  "
          f"{r['mean_reward']:>8.0f}  {r['pass_rate']:>6.0%}  {r['episodes']}")


## Cell 10: Interpret the live results

You should see a monotonic degradation from Lab → Harsh. On this cluster the policy typically reports:

- **Lab**: ~7,000+ reward, 100% pass — the policy was trained in sim, so sim-perfect is a ceiling
- **Factory floor**: slight reward drop, still ~100% pass — small noise barely affects a well-trained policy
- **Outdoor field**: noticeable reward and pass-rate drop (~60-70%)
- **Harsh (EOL)**: collapse — reward below 1,000, pass rate near 0

Exact numbers vary run-to-run because of noise randomness and the small (5-env) sample size. The **shape** is what matters: the policy is robust to mild conditions but cliffs hard under extreme perturbation. To map the cliff precisely, you need more conditions, which is what the pre-computed sweep below provides.

## Cell 11: Scale up — load the pre-computed 25-condition sweep

The live 4-scenario run shows the pattern. The **full deployment map** needs more resolution: 5 sensor levels × 5 motor levels = **25 conditions**, each with 20 parallel humanoids × 1,500 steps.

The full sweep was previously run on an identical cluster with more granular steps when adding noise. That run took **~14 minutes** on 4 A10G GPUs and produced 1,000 episodes worth of data. The results are saved in `data/sweep_results_full.json` so you can analyze them instantly below — no re-running required.

In [ ]:
RESULTS_PATH = "data/sweep_results_full.json"
with open(RESULTS_PATH) as f:
    sweep_data = json.load(f)

sweep_results = sweep_data["results"]
sweep_elapsed = sweep_data.get("elapsed", 0)
total_episodes = sweep_data.get("total_episodes",
                                sum(r["num_episodes"] for r in sweep_results))

obs_noises = sorted(set(r["obs_noise_std"] for r in sweep_results))
act_noises = sorted(set(r["action_noise_std"] for r in sweep_results))
obs_labels = [OBS_LABELS.get(o, str(o)) for o in obs_noises]
act_labels = [ACT_LABELS.get(a, str(a)) for a in act_noises]

# Build matrices for the heatmaps
pass_matrix   = np.zeros((len(obs_noises), len(act_noises)))
reward_matrix = np.zeros((len(obs_noises), len(act_noises)))
for r in sweep_results:
    oi = obs_noises.index(r["obs_noise_std"])
    ai = act_noises.index(r["action_noise_std"])
    pass_matrix[oi, ai]   = r["pass_rate"] * 100
    reward_matrix[oi, ai] = r["mean_reward"]

baseline = [r for r in sweep_results
            if r["obs_noise_std"] == 0 and r["action_noise_std"] == 0][0]
worst = min(sweep_results, key=lambda r: r["pass_rate"])

display(HTML(f"""
<div style="display:flex; gap:12px; margin:16px 0; flex-wrap:wrap;">
    <div style="flex:1; min-width:150px; background:#f0fdf4; border-radius:12px; padding:20px; text-align:center;">
        <div style="font-size:11px; color:#888; text-transform:uppercase; letter-spacing:1px;">Baseline reward</div>
        <div style="font-size:32px; font-weight:bold; color:#1D9E75;">{baseline['mean_reward']:,.0f}</div>
        <div style="font-size:12px; color:#666;">pass rate {baseline['pass_rate']:.0%} &middot; Lab conditions</div>
    </div>
    <div style="flex:1; min-width:150px; background:#fef2f2; border-radius:12px; padding:20px; text-align:center;">
        <div style="font-size:11px; color:#888; text-transform:uppercase; letter-spacing:1px;">Worst case</div>
        <div style="font-size:32px; font-weight:bold; color:#E24B4A;">{worst['mean_reward']:,.0f}</div>
        <div style="font-size:12px; color:#666;">pass {worst['pass_rate']:.0%} &middot; {OBS_LABELS.get(worst['obs_noise_std'],'')} + {ACT_LABELS.get(worst['action_noise_std'],'')}</div>
    </div>
    <div style="flex:1; min-width:150px; background:#f8f9fa; border-radius:12px; padding:20px; text-align:center;">
        <div style="font-size:11px; color:#888; text-transform:uppercase; letter-spacing:1px;">Sweep</div>
        <div style="font-size:32px; font-weight:bold; color:#333;">{len(sweep_results)}</div>
        <div style="font-size:12px; color:#666;">configs &middot; {total_episodes:,} episodes &middot; {sweep_elapsed:.0f}s</div>
    </div>
</div>
"""))


## Cell 12: Pass-rate heatmap — the deployment envelope

The pass-rate heatmap is the headline chart. Each cell shows the fraction of episodes where the humanoid successfully walked (reward > 3,000) under a specific combination of sensor noise and motor condition. Green is safe, red is unsafe, and the transition zone marks your deployment boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

cmap = mcolors.LinearSegmentedColormap.from_list(
    "deploy", ["#E24B4A", "#ef8b2c", "#facc15", "#86efac", "#1D9E75"])

im = ax.imshow(pass_matrix, cmap=cmap, vmin=0, vmax=100, aspect="auto")

ax.set_xticks(range(len(act_labels)))
ax.set_xticklabels(act_labels, fontsize=11, rotation=15, ha="right")
ax.set_yticks(range(len(obs_labels)))
ax.set_yticklabels(obs_labels, fontsize=11)

ax.set_xlabel("Motor condition", fontsize=13, fontweight="bold", labelpad=12)
ax.set_ylabel("Sensor quality",   fontsize=13, fontweight="bold", labelpad=12)
ax.set_title("Policy deployment readiness\nPass rate across 25 real-world conditions",
             fontsize=15, fontweight="bold", pad=16)

for i in range(len(obs_noises)):
    for j in range(len(act_noises)):
        val = pass_matrix[i, j]
        color = "white" if val < 35 or val > 80 else "#333"
        ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, label="Pass rate %", shrink=0.85, pad=0.02)
plt.tight_layout()
plt.show()


## Cell 13: Mean reward heatmap

Pass rate is a binary measure of whether the policy works and mean reward tells you *how well* it works when it does. You'd expect a smoother gradient here as even marginal conditions can still produce non-trivial rewards if the humanoid survives briefly before falling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

cmap2 = mcolors.LinearSegmentedColormap.from_list(
    "reward", ["#7f1d1d", "#E24B4A", "#fde68a", "#86efac", "#166534"])

im = ax.imshow(reward_matrix, cmap=cmap2, aspect="auto")

ax.set_xticks(range(len(act_labels)))
ax.set_xticklabels(act_labels, fontsize=11, rotation=15, ha="right")
ax.set_yticks(range(len(obs_labels)))
ax.set_yticklabels(obs_labels, fontsize=11)

ax.set_xlabel("Motor condition", fontsize=13, fontweight="bold", labelpad=12)
ax.set_ylabel("Sensor quality",   fontsize=13, fontweight="bold", labelpad=12)
ax.set_title("Mean episode reward\nHigher = better locomotion performance",
             fontsize=15, fontweight="bold", pad=16)

for i in range(len(obs_noises)):
    for j in range(len(act_noises)):
        val = reward_matrix[i, j]
        color = "white" if val < reward_matrix.mean() * 0.6 else "#333"
        ax.text(j, i, f"{val:,.0f}", ha="center", va="center",
                fontsize=12, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, label="Reward", shrink=0.85, pad=0.02)
plt.tight_layout()
plt.show()


## Cell 14: Degradation by noise axis

Collapse the 5×5 matrix onto each axis to see how much each failure mode contributes independently. Averaged over the other axis, this isolates **sensor noise impact** from **motor wear impact** — useful for engineering decisions like "invest in better IMUs" vs "invest in motor maintenance."

The dashed line marks the 3,000-reward deployment threshold (the same threshold used for pass-rate above).

In [ ]:
avg_by_obs = [np.mean([r["mean_reward"] for r in sweep_results if r["obs_noise_std"] == on])
              for on in obs_noises]
avg_by_act = [np.mean([r["mean_reward"] for r in sweep_results if r["action_noise_std"] == an])
              for an in act_noises]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors1 = ["#1D9E75" if v > 5000 else "#ef8b2c" if v > 3000 else "#E24B4A"
           for v in avg_by_obs]
bars1 = ax1.bar(obs_labels, avg_by_obs, color=colors1, edgecolor="white", linewidth=1.5)
ax1.axhline(y=3000, color="#E24B4A", linestyle="--", linewidth=1.5, label="Deploy threshold")
ax1.set_title("Sensor noise impact", fontsize=14, fontweight="bold")
ax1.set_ylabel("Mean reward", fontsize=12)
ax1.legend(fontsize=10)
for bar, val in zip(bars1, avg_by_obs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f"{val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax1.tick_params(axis="x", rotation=20, labelsize=10)

colors2 = ["#1D9E75" if v > 5000 else "#ef8b2c" if v > 3000 else "#E24B4A"
           for v in avg_by_act]
bars2 = ax2.bar(act_labels, avg_by_act, color=colors2, edgecolor="white", linewidth=1.5)
ax2.axhline(y=3000, color="#E24B4A", linestyle="--", linewidth=1.5, label="Deploy threshold")
ax2.set_title("Motor wear impact", fontsize=14, fontweight="bold")
ax2.set_ylabel("Mean reward", fontsize=12)
ax2.legend(fontsize=10)
for bar, val in zip(bars2, avg_by_act):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f"{val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.tick_params(axis="x", rotation=20, labelsize=10)

fig.suptitle("How real-world conditions degrade policy performance",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## Cell 15: Deployment decision

Turn the heatmaps into a one-line verdict. You bucket the 25 configs by pass rate:

- **Safe** (pass > 80%) — deployable as-is
- **Marginal** (40-80%) — deployable only in controlled environments
- **Fail** (< 40%) — re-train with domain randomization

And report the worst combined sensor+motor condition under which the policy still passed reliably. This is exactly the kind of data a robotics team would use in a pre-deployment review.

In [ ]:
safe_configs     = [r for r in sweep_results if r["pass_rate"] > 0.8]
marginal_configs = [r for r in sweep_results if 0.4 <= r["pass_rate"] <= 0.8]
fail_configs     = [r for r in sweep_results if r["pass_rate"] < 0.4]

max_safe_obs = max((r["obs_noise_std"]    for r in safe_configs), default=0)
max_safe_act = max((r["action_noise_std"] for r in safe_configs), default=0)
safe_env = f"{OBS_LABELS.get(max_safe_obs, str(max_safe_obs))} + {ACT_LABELS.get(max_safe_act, str(max_safe_act))}"

if len(safe_configs) > len(sweep_results) * 0.6:
    verdict = "Deploy with standard monitoring"
    verdict_color = "#1D9E75"
elif len(safe_configs) > len(sweep_results) * 0.3:
    verdict = "Deploy in controlled environments only"
    verdict_color = "#d97706"
else:
    verdict = "Additional training with domain randomization recommended"
    verdict_color = "#E24B4A"

display(HTML(f"""
<div style="background: linear-gradient(135deg, #f8f9fa, #fff); border: 1px solid #e0e0e0; border-radius: 12px; padding: 24px; margin: 16px 0;">
    <div style="font-size: 18px; font-weight: bold; margin-bottom: 16px;">Deployment decision</div>
    <div style="display: flex; gap: 16px; margin-bottom: 16px; flex-wrap: wrap;">
        <div style="flex: 1; min-width: 140px; padding: 12px; background: #f0fdf4; border-radius: 8px; text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #1D9E75;">{len(safe_configs)}</div>
            <div style="font-size: 12px; color: #666;">Safe configs (&gt;80%)</div>
        </div>
        <div style="flex: 1; min-width: 140px; padding: 12px; background: #fffbeb; border-radius: 8px; text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #d97706;">{len(marginal_configs)}</div>
            <div style="font-size: 12px; color: #666;">Marginal (40-80%)</div>
        </div>
        <div style="flex: 1; min-width: 140px; padding: 12px; background: #fef2f2; border-radius: 8px; text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #E24B4A;">{len(fail_configs)}</div>
            <div style="font-size: 12px; color: #666;">Fail (&lt;40%)</div>
        </div>
    </div>
    <div style="font-size: 14px; color: #444; line-height: 1.6;">
        <b>Safe operating envelope:</b> {safe_env}<br>
        <b>Verdict:</b> <span style="color:{verdict_color}; font-weight:bold;">{verdict}</span>
    </div>
</div>
"""))


## Cell 16: Cleanup

Release the GPU workers and remove the staged checkpoint so the notebook can be re-run cleanly. The committed `checkpoint_pretrained.pt` in the repo root is untouched and only the copy in `/mnt/cluster_storage/` is removed.

In [ ]:
if ray.is_initialized():
    ray.shutdown()
    print("Ray cluster released")

STAGED_CKPT_DIR = "/mnt/cluster_storage/checkpoints/humanoid"
if os.path.exists(STAGED_CKPT_DIR):
    shutil.rmtree(STAGED_CKPT_DIR)
    print(f"Removed {STAGED_CKPT_DIR}")

print("Cleanup complete — safe to re-run.")


## Conclusion

In this notebook you built a complete distributed robot policy sim-eval pipeline on **Ray + Isaac Lab on Anyscale**:

1. **Staged shared assets** — copied the pre-trained checkpoint to `/mnt/cluster_storage/` so every GPU worker reads the same file without per-node downloads
2. **Hosted the policy with Ray Serve** — `@serve.deployment class PolicyServer` keeps a single loaded copy of the policy on the cluster and serves every actor through a shared handle
3. **Wrapped Isaac Lab in a Ray Actor** — `@ray.remote(num_gpus=1) class SimActor` schedules one instance per GPU worker and amortizes the expensive Isaac Lab boot across many `run()` calls
4. **Fanned out live sim-eval** — instantiated 4 actors in parallel, dispatched one deployment scenario per actor, gathered results with `ray.get(futures)` — all 4 GPUs used concurrently
5. **Analyzed a 25-condition sweep** — loaded the pre-computed results (from `run_sweep.py`, which uses the stateless `@ray.remote def` pattern) and visualized the full deployment envelope as heatmaps, bar charts, and a decision panel

### What Ray on Anyscale gave you

The sim-eval logic and the policy network are pure Python + PyTorch. Ray on Anyscale handled:

- **GPU scheduling** — `@ray.remote(num_gpus=1)` declares the requirement, Ray places each actor on a free A10G
- **Parallel boot and fan-out** — 4 Isaac Lab instances boot simultaneously across 4 nodes; `ray.get(futures)` gathers results
- **Shared storage** — `/mnt/cluster_storage/` is visible to every worker with no extra plumbing
- **Scaling requires config changes, not code changes** — re-run this notebook on a 64-GPU cluster by changing `range(len(LIVE_CONFIGS))` to `range(64)` and adding more configs

### Three Ray patterns, one lesson

You used three Ray primitives side-by-side:

| Pattern | When to use | Used in |
|---|---|---|
| `@serve.deployment class` | One expensive object shared across many callers, concurrent requests | `PolicyServer` above |
| `@ray.remote class` (Actor) | Expensive per-worker setup, few long-lived workers, many cheap calls per worker | `SimActor` in the live sweep above |
| `@ray.remote def` (Function) | Cheap per-call setup, many independent calls, max parallelism | Pre-computed 25-config sweep (`run_sweep.py`) |

The same actor pattern also powers distributed PPO training in `train_general.py` — `SimWorker` is literally a `SimActor` used during rollout collection. Ray patterns generalize across the full train-to-deploy pipeline.

### The infrastructure is task-agnostic

The Ray patterns here don't depend on humanoid locomotion:

- **Swap the robot** — change `task="Isaac-Humanoid-Direct-v0"` to `"Isaac-Ant-Direct-v0"`, `"Isaac-Reach-Franka-Direct-v0"`, `"Isaac-Velocity-Rough-Anymal-C-Direct-v0"`, or any custom Isaac Lab Direct task. The `env.py` wrapper handles them all without changes.
- **Swap the policy** — replace `EvalPolicy` with any PyTorch `nn.Module` and update the state-dict load call. The actor-based fan-out pattern is untouched.
- **Swap the perturbation** — change what you add to obs/actions (domain randomization, adversarial noise, action dropout, partial observability). The evaluation loop is generic.

### Next steps

- **Scale up on Anyscale** — increase the live sweep to 25+ conditions (swap actors for functions as in `run_sweep.py`), scale to 16-64 GPUs with a bigger cluster config, and push `num_envs` per actor up to 512+ for tighter statistics
- **Swap the task** — re-point the `task=` argument at Ant, Franka, or ANYmal; the rest of the notebook stays the same
- **Add domain randomization** — randomize mass, friction, gravity, link lengths inside the env to train a more robust policy, then re-run this sweep to measure the deployment-envelope gain
- **Close the loop** — pipe the pass-rate map back into training as a curriculum signal, re-train under the conditions where the policy fails, re-evaluate, iterate

---

*Built for Anyscale. Works with any Isaac Lab task on GPU clusters.*
